In [ ]:
import json
from pathlib import Path
from itertools import chain
import numpy as np
from nd2 import ND2File
from calmutils.imageio.tiff_imagej import save_tiff_imagej, get_imagej_tiff_pixel_size, get_imagej_tiff_metadata
from calmutils.imageio.nd2_helpers import get_z_direction
from calmutils.stitching.transform_helpers import translation_matrix, scale_matrix
from tifffile import imread

try:
    from dask_image.ndinterp import affine_transform
    print('will use dask-image for image transformation')
except ImportError:
    from scipy.ndimage import affine_transform
    print('will use scipy for image transformation, consider dask-image for higher speed')

def world_coordinate_transform_to_pixel(transform_matrix, pixel_size):
    pixel_scale_mat = np.diag(list(pixel_size) + [1])
    mat = np.linalg.inv(pixel_scale_mat) @ transform_matrix @ pixel_scale_mat
    return mat

def drop_dimension_from_transform(tr, idx, fixed_value):

    # drop row of dimension idx
    tr_remove_row = np.delete(tr, idx, 0)
    # drop column of dimension idx
    tr_reduced = np.delete(tr_remove_row, idx, 1)

    # multiply coefficients for dropped dimension with fixed value, add to offset
    extra_off = tr_remove_row.T[idx][:-1] * fixed_value
    tr_reduced.T[-1,:-1] += extra_off

    return tr_reduced


SUPPORTED_SUFFIXES = (".tif", ".tiff", ".nd2")

# Correct Chromatic Aberrations for images

This notebook will use the channel-to-channel transformations estimated with ```chromatic_aberration_estimation{_elastix}.ipynb``` and apply them to new images.

We need:
1. the saved JSON transform information from the estimation recipes
2. image files to transform

**Input**
1. path to a directory containing nd2 files
2. path to the saved transforms
3. path to write aligned images to
4. **Parameters**: which channel to use as reference, optionally channel name map if the names differ in JSON and the metadata of new images

This recipe will produce:
* aligned images saved as multichannel TIFF files that can be read by ImageJ

In [ ]:
in_path = '/Users/david/Downloads/12_06_transfections_fixed/nanog-halo_BRD4_Oct4/'
transforms_path = '/Users/david/Downloads/channel_registration_multifile-3c.json'

image_subdirectory = ''
out_subdirectory = 'aligned'

# which channel the coordinates should be aligned to
reference_channel = '488-CSU-W1'
# reference_channel = 0

### Channel Renaming
# if the channel names in the JSON transform file and the new files differ
# e.g. if the OC names in NIS were different or images were resaved and just have channel 0, 1, ...,
# we have to rename the channels from the JSON file to match the ones in new images
# the channel alias map should have the form: name in JSON -> name in new images
# channel_aliases = {
#     '405 CSU-W1': '405-CSU-W1',
#     '488 CSU-W1': '488-CSU-W1',
#     '561 CSU-W1': '561-CSU-W1',
#     '640 CSU-W1': '640-CSU-W1'
# }

# if you do not want to rename the channels, just use an empty dict
channel_aliases = {}

# spinning disk data may have additional magnification of 1.5x
# leave at 1.0 unless you are sure you used the extra zoom
magnification = 1.0

In [ ]:
# make Path objects for input, output and parameter paths
out_path = Path(in_path) / out_subdirectory
in_path = Path(in_path) / image_subdirectory
transforms_path = Path(transforms_path)

In [ ]:
with open(transforms_path) as fd:
    transform_info = json.load(fd)

# transforms are saved as list of dicts containing channel pair and (flat) parameters
# build dict channel pair -> transform matrix
transforms = {}
for transform_info_i in transform_info['transforms']:

    transform_len = len(transform_info_i['parameters'])
    if transform_len == 16:
        tr = np.array(transform_info_i['parameters']).reshape(4,4)
    elif transform_len == 9:
        tr = np.array(transform_info_i['parameters']).reshape(3,3)

    # apply channel renaming if necessary
    channels = map(lambda c: channel_aliases[c] if c in channel_aliases else c, transform_info_i['channels'])

    transforms[tuple(channels)] = tr

in_files = sorted(chain.from_iterable(Path(in_path).glob(f'*{suffix}') for suffix in SUPPORTED_SUFFIXES))
in_files

In [ ]:
if not out_path.exists():
    out_path.mkdir()

for image_path in in_files:

    # skip hidden files
    if image_path.name.startswith('.'):
        continue

    print(f'aligning {image_path}')

    if image_path.suffix == '.nd2':
        # read all channels into dict of channel_name -> img
        images = {}
        with ND2File(image_path) as reader:

            is2d = 'Z' not in reader.sizes
            for i, channel in enumerate(reader.metadata.channels):
                if is2d:
                    img = np.array(reader.to_dask()[i])
                else:
                    img = np.array(reader.to_dask()[:,i])
                images[channel.channel.name.strip().replace(' ', '-')] = img
            # invert xyz voxel size to zyx to match img array
            pixel_size = np.array(reader.voxel_size()[::-1])

        z_fov = reader.sizes['Z'] * reader.voxel_size().z if not is2d else None

        z_direction = get_z_direction(image_path)

    # this should handle ZCYX TIFF files
    # TODO: support more?
    else:
        images = dict(enumerate(imread(image_path).swapaxes(0, 1)))
        pixel_size, pixel_unit = get_imagej_tiff_pixel_size(image_path)
        is2d = False
        z_direction = None

    calibration_fov = np.array(transform_info['field_of_view']) if 'field_of_view' in transform_info else None

    calibration_z_direction = transform_info['z_direction']

    # drop z for pixel size, calibration FOV if we are dealing with 2d data
    if is2d:
        pixel_size = pixel_size[1:]
        calibration_fov = calibration_fov[1:] if calibration_fov is not None else None

    images_aligned = {}
    for ch, image in images.items():

        # leave reference channel as-is
        if ch == reference_channel:
            images_aligned[ch] = image
            print(f'keep image of channel {ch} as-is (reference)')
            continue

        # catch no transform for given channel found -> leave as-is but warn.
        if not (((reference_channel, ch) in transforms) or ((ch, reference_channel) in transforms)):
            images_aligned[ch] = image
            print(f'WARNING: no transform for channel {ch} found, leaving as-is.')
            continue

         # NOTE: we want the inverse transform from ch to reference, i.e. the transform reference -> ch
        if (reference_channel, ch) in transforms:
            mat = transforms[(reference_channel, ch)]
        else:
            mat = np.linalg.inv(transforms[(ch, reference_channel)])

        # if we have 2d images but 3d transform, drop z dimension
        # NOTE: probably less accurate than transfrom estimated in 2d?
        if is2d and mat.shape == (4,4):
            z_fov_center = transform_info['field_of_view'][0]/2 if 'field_of_view' in transform_info else 0
            mat = drop_dimension_from_transform(mat, 0, z_fov_center)

        # get difference between calibration fov and image to correct
        # update transformation matrix to shift image by half diff (to center)
        # and then shift back after transform
        if calibration_fov is not None:
            img_fov = np.array(image.shape) * pixel_size
            offset_to_calibration = (calibration_fov - img_fov) / 2
            offset_mat = translation_matrix(offset_to_calibration)
            # right-to-left: shift, transform, shift back
            mat = np.linalg.inv(offset_mat) @ mat @ offset_mat

        z_flip_needed = z_direction is not None and (z_direction != calibration_z_direction)
        if z_flip_needed:
            flip_mat = translation_matrix([z_fov, 0, 0]) @ scale_matrix([-1, 1, 1])
            mat = np.linalg.inv(flip_mat) @ mat @ flip_mat

        mat = world_coordinate_transform_to_pixel(mat, pixel_size)

        image_transformed = affine_transform(image, mat, order=2)
        images_aligned[ch] = np.array(image_transformed)

        print(f'aligned image of channel {ch}')

    # save
    images_aligned_stacked = np.stack(list(images_aligned.values()))
    out_file = out_path / (image_path.stem + '_aligned.tif')
    axes_string = 'cyx' if is2d else 'czyx'

    # load existing metadata for ImageJ TIFF files (to keep LUTs, etc.)
    existing_meta = get_imagej_tiff_metadata(image_path) if image_path.suffix in ('.tif', '.tiff') else None

    save_tiff_imagej(out_file, images_aligned_stacked, axes=axes_string, pixel_size=pixel_size, distance_unit='micron', existing_metadata=existing_meta)

    print(f'finished aligning {image_path}')